# Filter DCT Vectors to Top-K Factors by Judge Score

Loads pre-computed lying scores from a source experiment, selects the top-K factors,
and overwrites `dct_vectors.pt` in the target experiment with only those K columns.

Run cells in order. Cell **[Preview]** shows exactly what will be kept before any file is written.

In [11]:
from pathlib import Path
import json
import shutil
import torch
import pandas as pd

# --- Configuration ---
# Experiment whose judge scores are used to rank factors
SCORE_EXPERIMENT  = "llama-3.1-8b-32samples-categorical"

# Experiment whose dct_vectors.pt will be filtered (overwritten)
TARGET_EXPERIMENT = "llama-3.1-8b-40prompts-categorical"

# Number of top factors to keep
TOP_K = 32

base = Path("dct_probes") if Path("dct_probes").exists() else Path(".")
score_path   = base / "experiments" / SCORE_EXPERIMENT  / "results" / "factor_lying_scores.json"
vectors_path = base / "experiments" / TARGET_EXPERIMENT / "vectors" / "dct_vectors.pt"
backup_path  = vectors_path.with_suffix(".pt.bak")

assert score_path.exists(),   f"Scores not found: {score_path}"
assert vectors_path.exists(), f"Vectors not found: {vectors_path}"

print(f"Score source:  {score_path}")
print(f"Target vectors: {vectors_path}")
print(f"Keeping top-{TOP_K} factors by lying score")

Score source:  experiments/llama-3.1-8b-32samples-categorical/results/factor_lying_scores.json
Target vectors: experiments/llama-3.1-8b-40prompts-categorical/vectors/dct_vectors.pt
Keeping top-32 factors by lying score


In [12]:
# Load and rank factors by lying score
with open(score_path) as f:
    factor_scores = json.load(f)

df = pd.DataFrame(factor_scores).sort_values("lying_score", ascending=False).reset_index(drop=True)
print(f"Total factors scored: {len(df)}")
print()
print(df[["factor_idx", "lying_score", "confidently_wrong_rate", "correct_rate", "n"]].head(TOP_K).to_string(index=True))

Total factors scored: 513

    factor_idx  lying_score  confidently_wrong_rate  correct_rate   n
0          374       0.4000                  0.4000        0.3000  10
1           52       0.4000                  0.4000        0.1000  10
2          236       0.3333                  0.3333        0.0000   9
3          350       0.3333                  0.3333        0.1111   9
4          142       0.3000                  0.3000        0.1000  10
5          201       0.3000                  0.3000        0.1000  10
6          285       0.3000                  0.3000        0.2000  10
7          329       0.3000                  0.3000        0.2000  10
8          227       0.3000                  0.3000        0.2000  10
9          439       0.3000                  0.3000        0.2000  10
10          83       0.3000                  0.3000        0.2000  10
11         204       0.3000                  0.3000        0.4000  10
12         163       0.3000                  0.3000        0.30

In [13]:
# [Preview] Show what will be kept — no files written yet
top_indices = df["factor_idx"].iloc[:TOP_K].tolist()

# Load from backup if it exists (i.e. notebook has been run before), so we
# always operate on the original unfiltered vectors rather than a previously
# filtered copy.
load_path = backup_path if backup_path.exists() else vectors_path
data = torch.load(load_path, weights_only=True, map_location="cpu")
V = data["V"]
U = data["U"]

print(f"Loaded vectors from: {load_path}")
print(f"Current V shape: {V.shape}  (d_model x num_factors)")
print(f"Current U shape: {U.shape}")
print(f"\nTop-{TOP_K} factor indices (by lying score):")
print(top_indices)
print(f"\nAfter filtering:")
print(f"  V will be: {V.shape[0]} x {TOP_K}")
print(f"  U will be: {U.shape[0]} x {TOP_K}")
print(f"\nBackup will be saved to: {backup_path}")
print(f"Target will be overwritten: {vectors_path}")

Loaded vectors from: experiments/llama-3.1-8b-40prompts-categorical/vectors/dct_vectors.pt
Current V shape: torch.Size([4096, 512])  (d_model x num_factors)
Current U shape: torch.Size([4096, 512])

Top-32 factor indices (by lying score):
[374, 52, 236, 350, 142, 201, 285, 329, 227, 439, 83, 204, 163, 471, 398, 167, 192, 168, 358, 359, 240, 66, 268, 280, 402, 4, 119, 131, 410, 130, 106, 406]

After filtering:
  V will be: 4096 x 32
  U will be: 4096 x 32

Backup will be saved to: experiments/llama-3.1-8b-40prompts-categorical/vectors/dct_vectors.pt.bak
Target will be overwritten: experiments/llama-3.1-8b-40prompts-categorical/vectors/dct_vectors.pt


In [14]:
# Write filtered vectors (backs up original first, only on first run)
if not backup_path.exists():
    shutil.copy2(vectors_path, backup_path)
    print(f"Backup saved to {backup_path}")
else:
    print(f"Backup already exists at {backup_path}, skipping")

filtered = {"U": U[:, top_indices], "V": V[:, top_indices]}
# Preserve any extra keys (e.g. scores, indices) if present
for key in data:
    if key not in ("U", "V"):
        orig = data[key]
        if isinstance(orig, torch.Tensor) and orig.dim() >= 1 and orig.shape[0] == V.shape[1]:
            filtered[key] = orig[top_indices]
        else:
            filtered[key] = orig

torch.save(filtered, vectors_path)

# Verify
check = torch.load(vectors_path, weights_only=True, map_location="cpu")
assert check["V"].shape == (V.shape[0], TOP_K), "Shape mismatch after save!"
print(f"Saved filtered vectors to {vectors_path}")
print(f"  V shape: {check['V'].shape}")
print(f"  U shape: {check['U'].shape}")
print(f"\nDone. Set NUM_FACTORS = {TOP_K} in dct_params.json for {TARGET_EXPERIMENT}.")

Backup saved to experiments/llama-3.1-8b-40prompts-categorical/vectors/dct_vectors.pt.bak
Saved filtered vectors to experiments/llama-3.1-8b-40prompts-categorical/vectors/dct_vectors.pt
  V shape: torch.Size([4096, 32])
  U shape: torch.Size([4096, 32])

Done. Set NUM_FACTORS = 32 in dct_params.json for llama-3.1-8b-40prompts-categorical.
